In [ ]:
!pip install optuna

In [ ]:
!pip install catboost

In [ ]:
pip install scikit-learn

In [114]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
import optuna
import os
from pathlib import Path

In [ ]:
pip install lightgbm 


In [ ]:
pip install xgboost

In [ ]:
pip install catboost


In [118]:

import lightgbm
import xgboost
import catboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer

In [119]:
df_path=".././data/data.csv"

In [120]:
df=pd.read_csv(df_path)

In [121]:
df.head(5)

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0


In [122]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sl_no           215 non-null    int64  
 1   gender          215 non-null    object 
 2   ssc_p           215 non-null    float64
 3   ssc_b           215 non-null    object 
 4   hsc_p           215 non-null    float64
 5   hsc_b           215 non-null    object 
 6   hsc_s           215 non-null    object 
 7   degree_p        215 non-null    float64
 8   degree_t        215 non-null    object 
 9   workex          215 non-null    object 
 10  etest_p         215 non-null    float64
 11  specialisation  215 non-null    object 
 12  mba_p           215 non-null    float64
 13  status          215 non-null    object 
 14  salary          148 non-null    float64
dtypes: float64(6), int64(1), object(8)
memory usage: 25.3+ KB


In [123]:
df.describe()

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
count,215.000000,215.000000,215.000000,215.000000,215.000000,215.000000,148.000000
mean,108.000000,67.303395,66.333163,66.370186,72.100558,62.278186,288655.405405
std,62.209324,10.827205,10.897509,7.358743,13.275956,5.833385,93457.452420
min,1.000000,40.890000,37.000000,50.000000,50.000000,51.210000,200000.000000
25%,54.500000,60.600000,60.900000,61.000000,60.000000,57.945000,240000.000000
50%,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,265000.000000
75%,161.500000,75.700000,73.000000,72.000000,83.500000,66.255000,300000.000000
max,215.000000,89.400000,97.700000,91.000000,98.000000,77.890000,940000.000000


In [124]:
df.fillna(0)

,sl_no,gender,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,0.0
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,211,M,80.60,Others,82.00,Others,Commerce,77.60,Comm&Mgmt,No,91.0,Mkt&Fin,74.49,Placed,400000.0
211,212,M,58.00,Others,60.00,Others,Science,72.00,Sci&Tech,No,74.0,Mkt&Fin,53.62,Placed,275000.0
212,213,M,67.00,Others,67.00,Others,Commerce,73.00,Comm&Mgmt,Yes,59.0,Mkt&Fin,69.72,Placed,295000.0
213,214,F,74.00,Others,66.00,Others,Commerce,58.00,Comm&Mgmt,No,70.0,Mkt&HR,60.23,Placed,204000.0


In [125]:
col_to_drop=['sl_no','salary','gender'] # i want my model to be unbiased wrt gender
df=df.drop(columns=col_to_drop,errors="ignore")

In [126]:

numerical_cols=df.select_dtypes(include=np.number).columns.to_list()
ordinal_categorical_col=None
nominal_categoricla_col=df.select_dtypes(include='object').columns.drop(['status']).to_list()
label_col=['status']

In [127]:
df.head(5)

,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status
0,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed
1,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed
2,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed
3,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed
4,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed


In [128]:
preprocess=ColumnTransformer([
    ('num',StandardScaler(),numerical_cols),
    ('cat',OneHotEncoder(handle_unknown='ignore'),nominal_categoricla_col)
])

In [129]:
df.head(5)

,ssc_p,ssc_b,hsc_p,hsc_b,hsc_s,degree_p,degree_t,workex,etest_p,specialisation,mba_p,status
0,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed
1,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed
2,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed
3,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed
4,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed


In [130]:
X_train,X_test,y_train,y_test=train_test_split(df.drop('status',axis=1),df['status'],test_size=0.2,random_state=42)

In [131]:
X_train = preprocess.fit_transform(X_train)
X_test= preprocess.transform(X_test)

In [132]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [133]:
def objective(trial):
    classifier_name = trial.suggest_categorical('classifier', ['XGBoost', 'LightGBM', 'CatBoost'])

    if classifier_name == 'XGBoost':
        params = {
            'n_estimators': trial.suggest_int('xgb_n_estimators', 100, 1000, step=100),
            'max_depth': trial.suggest_int('xgb_max_depth', 3, 10),
            'learning_rate': trial.suggest_float('xgb_learning_rate', 0.001, 0.1, log=True),
            'subsample': trial.suggest_float('xgb_subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.5, 1.0),
            'eval_metric': 'logloss' # For binary classification
        }
        model = xgboost.XGBClassifier(**params)
    elif classifier_name == 'LightGBM':
        params = {
            'n_estimators': trial.suggest_int('lgbm_n_estimators', 100, 1000, step=100),
            'max_depth': trial.suggest_int('lgbm_max_depth', 3, 10),
            'learning_rate': trial.suggest_float('lgbm_learning_rate', 0.001, 0.1, log=True),
            'subsample': trial.suggest_float('lgbm_subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('lgbm_colsample_bytree', 0.5, 1.0),
            'num_leaves': trial.suggest_int('lgbm_num_leaves', 2, 256),
            'objective': 'binary', # For binary classification
            'metric': 'binary_logloss' # For binary classification
        }
        model = lightgbm.LGBMClassifier(**params)
    else: # CatBoost
        params = {
            'iterations': trial.suggest_int('cb_iterations', 100, 1000, step=100),
            'depth': trial.suggest_int('cb_depth', 3, 10),
            'learning_rate': trial.suggest_float('cb_learning_rate', 0.001, 0.1, log=True),
            'l2_leaf_reg': trial.suggest_float('cb_l2_leaf_reg', 1e-8, 10.0, log=True),
            'loss_function': 'Logloss' # For binary classification
        }
        model = catboost.CatBoostClassifier(**params)

    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    return score

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) 

In [135]:
# Display the best trial found by Optuna
print("Number of finished trials:", len(study.trials))
print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

Number of finished trials: 50
Best trial:
  Value: 0.8605
  Params: 
    classifier: LightGBM
    lgbm_n_estimators: 300
    lgbm_max_depth: 10
    lgbm_learning_rate: 0.09886876976343996
    lgbm_subsample: 0.9645551038408261
    lgbm_colsample_bytree: 0.9420136229474116
    lgbm_num_leaves: 2


In [ ]:
best_params = study.best_trial.params
classifier_name = best_params['classifier']

if classifier_name == 'XGBoost':
    model_class = xgboost.XGBClassifier
    prefix = 'xgb_'
elif classifier_name == 'LightGBM':
    model_class = lightgbm.LGBMClassifier
    prefix = 'lgbm_'
else: # CatBoost
    model_class = catboost.CatBoostClassifier
    prefix = 'cb_'

# Filter parameters relevant to the best classifier and remove the prefix
filtered_params = {k.replace(prefix, ''): v for k, v in best_params.items() if k.startswith(prefix)}

# Instantiate and train the final model
final_model = model_class(**filtered_params)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_score = final_model.score(X_test, y_test)
print(f"Final model accuracy on test set: {final_score:.4f}")

[LightGBM] [Info] Number of positive: 117, number of negative: 55
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000267 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 246
[LightGBM] [Info] Number of data points in the train set: 172, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.680233 -> initscore=0.754841
[LightGBM] [Info] Start training from score 0.754841
Final model accuracy on test set: 0.8605


In [137]:
classifier_name = best_params['classifier']
classifier_name

'LightGBM'

In [138]:
type(final_model)

lightgbm.sklearn.LGBMClassifier

In [ ]:
print(params)

<bound method CatBoost.get_all_params of CatBoostClassifier(depth=10, iterations=1000, l2_leaf_reg=5.00884054101728, learning_rate=0.07621274053762664)>


In [140]:
import pickle
os.makedirs('models',exist_ok=True)
with open('models/final_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

In [141]:
def get_model_params_dict(model) -> dict:
    """Extracts hyperparameters as a clean dictionary from any framework."""
    model_class_name = model.__class__.__name__

    # 1. CatBoost
    if "CatBoost" in model_class_name:
        return model.get_all_params()  # <-- Don't forget the parentheses!

    # 2. XGBoost
    elif "XGB" in model_class_name:
        return model.get_params()

    # 3. LightGBM
    elif "LGBM" in model_class_name:
        return model.get_params()

    else:
        raise ValueError(f"Unsupported model type: {model_class_name}")


In [148]:
import json

In [149]:
params=get_model_params_dict(final_model)

with open('json_params.json','w') as f:
    json.dump(params,f,indent=4)